In [0]:
%run ./00_config

In [0]:
#importing libraries 
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.model_selection import cross_val_score

1.  Read and Dispaly data 

In [0]:
Goldd = spark.read.format("delta").load(f"{Gold_base}/Gold_Borough_Year")
pdf = Goldd.toPandas().dropna()

display(Goldd.limit(10))


2. Correlations 

In [0]:
corr_FLY_crime = pdf["fly_tipping_incidents"].corr(pdf["minor_crime_count"])
Corre_pull_crime = pdf["neighbour_all_together"].corr(pdf["minor_crime_count"])
Corr_help_crime = pdf["people_help_avail"].corr(pdf["minor_crime_count"])
Corr_Belong_crime = pdf["local_area_belong_index"].corr(pdf["minor_crime_count"])

print("Fly-tipping vs minor crime:", corr_FLY_crime)
print("Neighbourhood All together vs crime:", Corre_pull_crime)
print("Help Availablity  vs crime:", Corr_help_crime)
print("Belonging Index vs crime:", Corr_Belong_crime)

3. Regression 


In [0]:
X = pdf[[
    "fly_tipping_incidents",
    "neighbour_all_together",
    "people_help_avail",
    "local_area_belong_index",
    "formal_volunt_percentage",
    "lonely_percentage"
]]

y = pdf["minor_crime_count"]

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

r2= r2_score(y, pred)
MAE = mean_absolute_error(y, pred)
#cross validation 
cv_score = cross_val_score(model, X, y, cv=5, scoring="r2")
cv_r2_mean = cv_score.mean()
cv_r2_std = cv_score.std()

print("Regression R²:", r2)
print("Regression MAE:", MAE)
print("Cross-validation R² mean:", cv_r2_mean)
print("Cross-validation R² std:", cv_r2_std)



In [0]:
# Coefficients
for col, coef in zip(X.columns, model.coef_):
    print(col, coef)

4. MLflow Logging 

In [0]:
# Intercept
print("Intercept:", model.intercept_)
with mlflow.start_run(run_name="broken_windows_cst_analysis"):
    mlflow.log_metric("r2", float(r2))
    mlflow.log_metric("mae", float(MAE))
    mlflow.log_metric("cv_r2_mean", float(cv_r2_mean))
    mlflow.log_metric("cv_r2_std", float(cv_r2_std))
    mlflow.log_metric("corr_fly_tipping_crime", float(corr_FLY_crime))
    mlflow.log_metric("corr_neighbour_all_together_crime", float(Corre_pull_crime))
    mlflow.log_metric("corr_help_available_crime", float(Corr_help_crime))
    mlflow.log_metric("corr_belonging_crime", float(Corr_Belong_crime))

print("MLflow is logged")

5. Visulizations 

In [0]:
#fly-tipping vs crime
plt.figure(figsize=(8,6))
plt.scatter(pdf["fly_tipping_incidents"], pdf["minor_crime_count"])
plt.xlabel("Fly-tipping incidents")
plt.ylabel("Minor crime count")
plt.title("Fly-tipping vs Low-level Crime")
plt.grid(True)
plt.show()

In [0]:
#community strength vs crime
plt.figure(figsize=(8,6))
plt.scatter(pdf["neighbour_all_together"], pdf["minor_crime_count"])
plt.xlabel("Neighbourhood All together (%)")
plt.ylabel("Minor crime count")
plt.title("Community Strength vs Low-level Crime")
plt.grid(True)
plt.show()

In [0]:
#trend year by year
trend = pdf.groupby("financial_year")[["minor_crime_count", "fly_tipping_incidents"]].sum().reset_index()

plt.figure(figsize=(9,6))
plt.plot(trend["financial_year"], trend["minor_crime_count"], marker="o", label="Minor crime")
plt.xticks(rotation=45)
plt.xlabel("Financial Year")
plt.ylabel("Count")
plt.title("Minor crime trend over period")
plt.grid(True)
plt.legend()
plt.show()

In [0]:
pdf.describe()